# Scheduling example


In [ ]:
import torch
import open_clip
from scenes import SpringScene, SciFiRobotScene, CarScene, BlenderManScene, HouseScene, DinoScene
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook

In [ ]:
torch_precision = torch.float32
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
lr = 0.04
n_iter = 200

In [ ]:
global_seed = 2 # Can be None

In [ ]:
clip_model_name = 'ViT-B-16-SigLIP-512'
clip_pretrained = 'webli'
model, _, preprocess_eval = open_clip.create_model_and_transforms(clip_model_name, pretrained=clip_pretrained, device=device)
tokenizer = open_clip.get_tokenizer(clip_model_name)
model.eval()

In [ ]:
from scenes import SpringScene, SciFiRobotScene, CarScene, BlenderManScene, HouseScene, DinoScene, FlowerPotScene, RedCarScene, CandleScene
scene = FlowerPotScene()

In [ ]:
from torchvision.transforms.v2 import RandomChoice, RandomPerspective, RandomResizedCrop, RandomHorizontalFlip, GaussianBlur, Identity, Transform
# Here's an example of a callback that can iteratively blur the rendered images based on the epoch (reducing the blur over time)
def iterative_blurring_callback(epoch:int) -> Transform:
    def fit(value, old_min, old_max, new_min, new_max):
        return new_min + (value - old_min) * (new_max - new_min) / (old_max - old_min)
    def get_closest_odd_number(n: float) -> int:
        n = int(round(n))
        if n % 2 == 0:
            n += 1
        return n
    max_blur = 60.0
    min_blur = 0.0
    max_kernel_size = 71
    min_kernel_size = 5
    kernel_size = get_closest_odd_number(fit(epoch, 0, n_iter, max_kernel_size, min_kernel_size))
    blur_amount = fit(epoch, 0, n_iter, max_blur, min_blur)
    if blur_amount <= 0:
        return Identity()
    return GaussianBlur(kernel_size=(kernel_size, kernel_size), sigma=blur_amount)

In [ ]:
# Example HSV scheduling callback
def hsv_schedule(epoch:int) -> str:
    if epoch < n_iter * 0.25: # Only optimize the brightness of lights for the first quarter of training, etc.
        return "v"
    elif epoch < n_iter * 0.5:
        return "sv"
    else:
        return "hsv"

In [ ]:
initial_text = "flat, unappealing lighting"
target_text = "lit at noon"

In [ ]:
from losses.clip import CLIPCosineSimilarity, CLIPDirectionalCosineSimilarity
from utils.train import train_with_criterion

color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())

clip_loss = CLIPDirectionalCosineSimilarity(
    initial_text=initial_text,
    target_text=target_text,
    initial_image = scene.get_combined_image(color_space_converter).permute(2, 0, 1),
    model=model,
    tokenizer=tokenizer,
    device=device,
    preprocess=preprocess_eval,
    always_prenormalize_vectors=True,
)

# clip_loss = CLIPCosineSimilarity(
#     text=target_text,
#     model=model,
#     tokenizer=tokenizer,
#     device=device,
#     preprocess=preprocess_eval,
# )

criterion = clip_loss

size = model.visual.preprocess_cfg['size'] or (224, 224)

train_with_criterion(
    scene,
    lr, n_iter, criterion,
    starting_multiplier_std=(0.1, 0.1, 0.1),
    output_subdirectory_name="scheduling_example",
    n_results=1,
    torch_precision=torch_precision,
    augmentation=RandomChoice([RandomResizedCrop(size=size, scale=(0.5, 1.0), antialias=True)]),
    augmentation_callback=iterative_blurring_callback,
    parameters_stored_as_hsv=True,
    hsv_callback=hsv_schedule,
    render_color_space_converter=color_space_converter,
    require_physically_plausible_multipliers=True,
    title_prefix="Scheduling with CLIP Loss",
    device=device,
    save_every=40,
    model_name=clip_model_name,
    pretrained_source=clip_pretrained,
    seed=global_seed,
    show_images_after_augmentation=True
)